# 04 — Account Segmentation

Four-class account segmentation:
1. **Eligible - No Risk**: Growing, low utilization, clean history
2. **Eligible - With Risk**: Active but stress signals present
3. **No Increase Needed**: Stable, no growth demand
4. **Non-Performing**: Delinquent, fraud-flagged, or inactive

Approach: Rule-based labels → Random Forest classifier

In [32]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_pipeline
from src import features as feat
from src.models import segmentation as seg

pd.set_option('display.max_columns', 50)

In [33]:
con, counts = load_pipeline(verbose=True)

Loading CSV files into DuckDB…
  ✓ account_dim: 18,070 rows
  ✓ statement_fact: 658,228 rows
  ✓ transaction_fact: 493,336 rows
  ✓ wrld_stor_tran_fact: 1,053,854 rows
  ✓ syf_id: 18,070 rows
  ✓ rams_batch_cur: 96,799 rows
  ✓ fraud_claim_case: 77 rows
  ✓ fraud_claim_tran: 202 rows
  ✓ transaction_base (union): 1,547,190 rows

Verifying row counts…
  ✓ account_dim: 18,070 rows (OK)
  ✓ statement_fact: 658,228 rows (OK)
  ✓ transaction_fact: 493,336 rows (OK)
  ✓ wrld_stor_tran_fact: 1,053,854 rows (OK)
  ✓ transaction_base: 1,547,190 rows (OK)
  ✓ syf_id: 18,070 rows (OK)
  ✓ rams_batch_cur: 96,799 rows (OK)
  ✓ fraud_claim_case: 77 rows (OK)
  ✓ fraud_claim_tran: 202 rows (OK)


## 1. Build Customer Base & Apply Rule-Based Labels

In [34]:
customer_base = feat.build_customer_base(con)
print(f"customer_base: {customer_base.shape}")

customer_base: (18070, 55)


In [35]:
# Apply rule-based segmentation
customer_segmented = seg.assign_rule_based_segments(customer_base)

print("\n=== Segment Distribution ===")
dist = customer_segmented['segment_name'].value_counts()
print(dist)
print(f"\nTotal: {len(customer_segmented):,}")


=== Segment Distribution ===
segment_name
No Increase Needed      12607
Eligible - No Risk       3471
Eligible - With Risk     1828
Non-Performing            164
Name: count, dtype: int64

Total: 18,070


In [37]:
# Segment distribution bar chart (with credit exposure)
seg_summary = customer_segmented.groupby('segment_name').agg(
    account_count=('current_account_nbr', 'count'),
    total_credit_exposure=('cu_crd_line', lambda x: pd.to_numeric(x, errors='coerce').sum()),
    avg_utilization=('ca_current_utilz', lambda x: pd.to_numeric(x, errors='coerce').mean()),
    avg_balance=('cu_cur_balance', lambda x: pd.to_numeric(x, errors='coerce').mean()),
).reset_index()

fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Account Count by Segment', 'Total Credit Exposure by Segment'
])

colors = [seg.SEGMENT_COLORS.get(s, '#999') for s in seg_summary['segment_name']]

fig.add_trace(go.Bar(
    x=seg_summary['segment_name'], y=seg_summary['account_count'],
    marker_color=colors, name='Accounts'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=seg_summary['segment_name'], y=seg_summary['total_credit_exposure'],
    marker_color=colors, name='Credit Exposure'
), row=1, col=2)

fig.update_layout(title_text='Segment Distribution with Total Credit Exposure',
                  showlegend=False, height=500, width=1000, template='plotly_white')
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Total Credit Line ($)', row=1, col=2)
fig.write_image("outputs/figures/segment_distribution.png", scale=2)
fig.show()

print("\nSegment Summary:")
print(seg_summary.to_string(index=False))


Segment Summary:
        segment_name  account_count  total_credit_exposure  avg_utilization  avg_balance
  Eligible - No Risk           3471               30743189        22.939499  1740.444898
Eligible - With Risk           1828                7943320        76.752735  3275.623807
  No Increase Needed          12607               73551781         1.378520   580.701889
      Non-Performing            164                 801758        27.878049   811.315549


## 2. Optional: K-Means Clustering Feature

In [38]:
# Add K-Means cluster as additional feature
customer_clustered, kmeans_model, cluster_scaler = seg.add_kmeans_cluster_feature(
    customer_segmented, n_clusters=6
)
print(f"\nK-Means Cluster Distribution:")
print(customer_clustered['cluster_id'].value_counts().sort_index())


K-Means Cluster Distribution:
cluster_id
0     2734
1    10539
2       90
3      899
4      780
5     3028
Name: count, dtype: int64


## 3. Train Random Forest Classifier

In [39]:
# Add cluster_id to classification features
clf_features = seg.CLASSIFICATION_FEATURES + ['cluster_id']

model, report, cm, test_df, used_features = seg.train_segmentation_classifier(
    customer_clustered,
    feature_cols=clf_features,
    test_size=0.25,
    save_path='outputs/saved_models/rf_segmentation.joblib'
)


  Random Forest Segmentation (weighted F1: 0.9924)
                      precision    recall  f1-score   support

      Non-Performing       0.70      0.77      0.73        39
  No Increase Needed       1.00      0.99      1.00      3029
Eligible - With Risk       0.99      1.00      1.00       457
  Eligible - No Risk       0.99      1.00      0.99       868

            accuracy                           0.99      4393
           macro avg       0.92      0.94      0.93      4393
        weighted avg       0.99      0.99      0.99      4393

  → Model saved to outputs/saved_models/rf_segmentation.joblib


## 4. Confusion Matrix

In [40]:
segment_names = [seg.SEGMENTS[i] for i in sorted(seg.SEGMENTS.keys())]

fig = px.imshow(
    cm,
    labels=dict(x="Predicted", y="Actual", color="Count"),
    x=segment_names, y=segment_names,
    color_continuous_scale='Blues',
    title='Segmentation — Confusion Matrix',
    text_auto=True,
    width=700, height=600
)
fig.update_layout(template='plotly_white')
fig.write_image("outputs/figures/segmentation_confusion_matrix.png", scale=2)
fig.show()

## 5. Feature Importance

In [41]:
importance_df = pd.DataFrame({
    'feature': used_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig = px.bar(importance_df.tail(15), x='importance', y='feature', orientation='h',
             title='Random Forest Segmentation — Top 15 Feature Importances',
             color='importance', color_continuous_scale='Greens')
fig.update_layout(height=500, width=700, template='plotly_white', yaxis_title='')
fig.write_image("outputs/figures/segmentation_feature_importance.png", scale=2)
fig.show()

## 6. Segment Profiles

In [42]:
# Key metrics by segment
profile_cols = ['cu_bhv_scr', 'cu_crd_bureau_scr', 'ca_current_utilz',
                'avg_monthly_sales_6m', 'cu_nbr_days_dlq', 'ca_nsf_count_lst_12_months',
                'delinquent_cycle_count', 'cu_crd_line']

for col in profile_cols:
    if col in customer_segmented.columns:
        customer_segmented[col] = pd.to_numeric(customer_segmented[col], errors='coerce')

segment_profile = customer_segmented.groupby('segment_name')[profile_cols].mean().round(2)
print("\n=== Average Feature Values by Segment ===")
print(segment_profile.to_string())


=== Average Feature Values by Segment ===
                      cu_bhv_scr  cu_crd_bureau_scr  ca_current_utilz  avg_monthly_sales_6m  cu_nbr_days_dlq  ca_nsf_count_lst_12_months  delinquent_cycle_count  cu_crd_line
segment_name                                                                                                                                                                 
Eligible - No Risk        578.51             757.55             22.94               1096.47              0.0                         0.0                    0.05      8857.16
Eligible - With Risk      561.82             663.75             76.75                434.97              6.7                        0.04                    0.80      4345.36
No Increase Needed        413.03             738.99              1.38                366.63              0.3                        0.01                    0.12       5834.2
Non-Performing            510.59              518.9             27.88                 7

## 7. Save Segmented Data

In [43]:
customer_segmented.to_parquet('outputs/predictions/customer_segmented.parquet', index=False)
print(f"Saved {len(customer_segmented)} segmented accounts")

Saved 18070 segmented accounts
